In [2]:
import requests
from bs4 import BeautifulSoup
import os
import time
from datetime import datetime

In [ ]:

URL = "https://en.wikipedia.org/wiki/Lionel_Messi"

headers = {
    "User-Agent": "Aula-CPA"
}

#Cria as pastas
output_dir = "data"
html_dir = os.path.join(output_dir, "html")

os.makedirs(output_dir, exist_ok=True)

#Criar dicionário para armazenar os metadados
metadata = {}

#limpar html antigo
import shutil
if os.path.exists(html_dir):
    shutil.rmtree(html_dir)

os.makedirs(html_dir)

#baixar HTML principal
response = requests.get(URL, headers=headers)
response_soup = BeautifulSoup(response.text, "html.parser")

main_html_path = os.path.join(html_dir, "main.html")

with open(main_html_path, "w", encoding="utf-8") as f:
    f.write(response_soup.prettify())

metadata["main.html"] = {
    "url": URL,
    "timestamp": datetime.now()
}

print("HTML principal salvo")

#extrair links A PARTIR do HTML salvo
with open(main_html_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if href.startswith("/wiki/") and ":" not in href:
        full_link = "https://en.wikipedia.org" + href
        links.add(full_link)

links = list(links)

print(f"{len(links)} links encontrados")

#baixar páginas dos links
for i, link in enumerate(links): #se nao quiser baixar os 2000 links, pode usar (links[:10]) por exemplo
    try:
        r = requests.get(link, headers=headers)
        r_soup = BeautifulSoup(r.text, "html.parser")

        file_name = f"page_{i}.html"
        file_path = os.path.join(html_dir, f"page_{i}.html")

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(r_soup.prettify())

        metadata[file_name] = {
            "url": link,
            "timestamp": datetime.now()
        }

        time.sleep(1)

    except:
        continue

print("Download concluído")

HTML principal salvo
2051 links encontrados
Download concluído


In [ ]:
import pandas as pd

#Garante o caminho
html_dir = os.path.join(output_dir, "html")

#Sempre reinicia (evita duplicação)
data_pages = []
data_images = []

for file in os.listdir(html_dir):
    path = os.path.join(html_dir, file)

    if not os.path.isfile(path):
        continue

    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    #título
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None

    #conteúdo principal
    content = soup.find("div", {"id": "mw-content-text"})
    first_paragraph = None

    if content:
        paragraphs = content.find_all("p", recursive=True)

        for p in paragraphs:
            text = p.get_text(strip=True)
            if text:
                first_paragraph = text
                break

    #imagens
    images = [
        #imagens inuteis como de interface permanecem static
        "https:" + img["src"] if img["src"].startswith("//") else img["src"]
        for img in soup.find_all("img")
        if img.get("src")
    ]

    #links internos
    internal_links = [
        a["href"] for a in soup.find_all("a", href=True)
        if a["href"].startswith("/wiki/")
    ]

    link_principal = metadata[file]["url"]
    timestamp = metadata[file]["timestamp"]

    #salva uma vez só
    data_pages.append({
        "title": title,
        "first_paragraph": first_paragraph,
        "link_principal": link_principal,
        "internal_links": ["https://en.wikipedia.org" + link for link in internal_links],
        "timestamp": timestamp
    })

    #imagens
    for img in images:
        data_images.append({
            "image_url": img
        })

In [13]:
df_pages = pd.DataFrame(data_pages)
df_images = pd.DataFrame(data_images)

df_pages.to_csv("data/pages.csv",sep=";", index=False)
df_images.to_csv("data/images.csv", sep=";", index=False)

In [14]:
df_pages.head(20)

,title,first_paragraph,link_principal,internal_links,timestamp
0,Lionel Messi,"Lionel Andrés""Leo""Messi[note 1](born 24 June 1...",https://en.wikipedia.org/wiki/Lionel_Messi,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:07.146414
1,Rogério Ceni,Rogério Mücke Ceni(Brazilian Portuguese:[ʁoˈʒɛ...,https://en.wikipedia.org/wiki/Rog%C3%A9rio_Ceni,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:12.679883
2,Copa América awards,This is a list ofawardspresented to players an...,https://en.wikipedia.org/wiki/Copa_Am%C3%A9ric...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:14.900703
3,TimePerson of the Year,"Person of the Year, calledMan of the YearorWom...",https://en.wikipedia.org/wiki/Time_Person_of_t...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:17.044910
4,Garry Birtles,Garry Birtles(born 27 July 1956) is an English...,https://en.wikipedia.org/wiki/Garry_Birtles,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:18.628276
5,Diego Milito,Diego Alberto Milito(born 12 June 1979) is an ...,https://en.wikipedia.org/wiki/Diego_Milito,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:21.109887
6,2017 FIFA U-20 World Cup,The2017 FIFA U-20 World Cupwas the 21st editio...,https://en.wikipedia.org/wiki/2017_FIFA_U-20_W...,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:24.303958
7,Mike Krzyzewski,Michael William Krzyzewski(US:/ʃɪˈʒɛfski/shizh...,https://en.wikipedia.org/wiki/Mike_Krzyzewski,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:26.879527
8,Liolaemus messii,Liolaemus messiiis a species oflizardin the fa...,https://en.wikipedia.org/wiki/Liolaemus_messii,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:28.413046
9,Paul Breitner,Paul Breitner(German pronunciation:[ˈpaʊlˈbʁaɪ...,https://en.wikipedia.org/wiki/Paul_Breitner,"[https://en.wikipedia.org/wiki/Main_Page, http...",2026-04-09 09:40:30.228015
